In [3]:
# =============================================================
# NOTEBOOK 02 – EDA & GIẢM CHIỀU DỮ LIỆU
# Môn: Học Máy MAT3533
# Đề tài: Hồi quy – IBM HR Analytics Employee Attrition
# Thực hiện: Phan Thanh Lâm
# Tuần: 1 | Ngày T5 – T6
# =============================================================

# ─── IMPORTS ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy import stats as scipy_stats
import os
import warnings
warnings.filterwarnings('ignore')

# Style chung cho biểu đồ đẹp
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

# Màu sắc thống nhất cho các biểu đồ
COLORS = {
    'purple': '#534AB7', 
    'teal': '#1D9E75', 
    'coral': '#E24B4A',
    'amber': '#EF9F27', 
    'pink': '#D4537E',
    'jl': {1: '#E24B4A', 2: '#EF9F27', 3: '#534AB7', 4: '#1D9E75', 5: '#D4537E'},
    'income': {'Low\n(<4K)': '#E24B4A', 'Mid\n(4-7K)': '#EF9F27', 'High\n(>7K)': '#1D9E75'},
}

# ─── PATHS (Windows) ───────────────────────────────────────
CLEANED_PATH = '../data/processed/df_cleaned.csv'
SCALED_PATH = '../data/processed/df_scaled.csv'

# Tạo thư mục output nếu chưa có
os.makedirs('data/processed', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)

print("=" * 60)
print("NOTEBOOK 02 – EDA & GIẢM CHIỀU DỮ LIỆU")
print("=" * 60)

# ─── LOAD DATA ─────────────────────────────────────────────
print("\n📂 Đang đọc dữ liệu...")
df_cleaned = pd.read_csv(CLEANED_PATH)
df_scaled = pd.read_csv(SCALED_PATH)

TARGET = 'MonthlyIncome'
feat_cols = [c for c in df_scaled.columns if c != TARGET and c not in ['Attrition', 'EmployeeCount', 'Over18', 'StandardHours']]
X = df_scaled[feat_cols].values
y = df_cleaned[TARGET].values
job_level = df_cleaned['JobLevel'].values

print(f"✓ Loaded: X={X.shape}, y={y.shape}")
print(f"✓ Target range: [{y.min():.0f}, {y.max():.0f}], mean={y.mean():.0f}")

# ═══════════════════════════════════════════════════════════
# BƯỚC 1 – PHÂN TÍCH MISSING & TARGET DISTRIBUTION
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 1 – MISSING VALUES & TARGET DISTRIBUTION")
print("=" * 60)

# Kiểm tra missing
completeness = (df_cleaned.notna().mean() * 100).sort_values()
skew_val = scipy_stats.skew(y)
print(f"✓ Completeness min: {completeness.min():.1f}% → dataset rất sạch")
print(f"✓ MonthlyIncome skewness: {skew_val:.3f} (lệch phải – phân phối lương điển hình)")

# Hình 1: Missing & Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hình 1: Missing Values & Phân phối MonthlyIncome', fontsize=13, fontweight='bold')

# Biểu đồ completeness
axes[0].barh(range(len(completeness[-20:])), completeness[-20:], color=COLORS['teal'], alpha=0.8)
axes[0].set_yticks(range(len(completeness[-20:])))
axes[0].set_yticklabels([c[:25] for c in completeness[-20:].index], fontsize=7)
axes[0].set_xlabel('Completeness (%)')
axes[0].set_title('Tỉ lệ dữ liệu đầy đủ (top 20 features)')
axes[0].axvline(100, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlim(95, 101)

# Histogram target
axes[1].hist(y, bins=40, color=COLORS['purple'], alpha=0.75, edgecolor='white', linewidth=0.4)
axes[1].axvline(np.mean(y), color='red', linestyle='--', lw=1.5, label=f'Mean={np.mean(y):.0f}')
axes[1].axvline(np.median(y), color='orange', linestyle='--', lw=1.5, label=f'Median={np.median(y):.0f}')
axes[1].set_xlabel('MonthlyIncome (USD)')
axes[1].set_ylabel('Tần số')
axes[1].set_title('Phân phối MonthlyIncome (Target)')
axes[1].legend(fontsize=9)
axes[1].text(0.98, 0.95, f'Skewness: {skew_val:.2f}', transform=axes[1].transAxes,
             ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='#FFF3CD', alpha=0.8))

plt.tight_layout()
plt.savefig('outputs/figures/fig01_missing_target.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig01_missing_target.png")

# ═══════════════════════════════════════════════════════════
# BƯỚC 2 – OUTLIER BOXPLOTS
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 2 – OUTLIER BOXPLOTS (Phát hiện outliers bằng IQR)")
print("=" * 60)

key_numeric = ['Age', 'DailyRate', 'DistanceFromHome', 'HourlyRate', 'MonthlyIncome',
               'MonthlyRate', 'NumCompaniesWorked', 'TotalWorkingYears',
               'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion']

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Hình 2: Boxplot – Phát hiện Outliers (IQR Method)', fontsize=13, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(key_numeric):
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df_cleaned[col] < Q1 - 1.5*IQR) | (df_cleaned[col] > Q3 + 1.5*IQR)).sum()
    color = COLORS['coral'] if n_out > 0 else COLORS['teal']
    axes[i].boxplot(df_cleaned[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.4),
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='o', markersize=2, alpha=0.3, color=color))
    axes[i].set_title(f'{col}\n({n_out} outliers, {n_out/len(df_cleaned)*100:.1f}%)', fontsize=9)
    axes[i].set_xticklabels([])

for j in range(len(key_numeric), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/figures/fig02_outliers_boxplot.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig02_outliers_boxplot.png")

# ═══════════════════════════════════════════════════════════
# BƯỚC 3 – PCA ANALYSIS
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 3 – PCA (Principal Component Analysis)")
print("=" * 60)

# Fit PCA để phân tích explained variance
pca_full = PCA()
pca_full.fit(X)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

print("\nSố components cần để đạt ngưỡng variance:")
for pct in [0.80, 0.90, 0.95]:
    n_comp = np.argmax(cumulative >= pct) + 1
    print(f"  {pct*100:.0f}% → {n_comp} components ({n_comp/X.shape[1]*100:.0f}% số features)")

# PCA với 10 components để visualize
n_vis = 10
pca_vis = PCA(n_components=n_vis)
X_pca = pca_vis.fit_transform(X)

# Feature loadings
loadings = pd.DataFrame(
    pca_vis.components_.T,
    index=feat_cols,
    columns=[f'PC{i+1}' for i in range(n_vis)]
)

print("\nTop 5 features đóng góp nhiều nhất vào PC1:")
top_pc1 = loadings['PC1'].abs().sort_values(ascending=False).head(5)
for feat, val in top_pc1.items():
    print(f"  {feat:<30}: |loading| = {val:.4f}")

# Hình 3: Scree plot + Feature loadings
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hình 3: PCA – Explained Variance & Feature Loadings', fontsize=13, fontweight='bold')

# Scree plot
n_show = 20
axes[0].bar(range(1, n_show+1), explained[:n_show]*100, color=COLORS['purple'], alpha=0.75)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot')
ax0b = axes[0].twinx()
ax0b.plot(range(1, n_show+1), cumulative[:n_show]*100, 'r-o', ms=4, lw=2)
ax0b.set_ylabel('Cumulative (%)', color='red')
ax0b.tick_params(axis='y', labelcolor='red')
for pct, col in [(80, 'green'), (90, 'orange'), (95, 'red')]:
    ax0b.axhline(pct, color=col, linestyle='--', alpha=0.5, lw=1, label=f'{pct}%')
ax0b.legend(loc='center right', fontsize=8)

# Feature loadings
top_pc1_plot = loadings['PC1'].abs().sort_values(ascending=False).head(12)
top_pc2_plot = loadings['PC2'].abs().sort_values(ascending=False).head(12)
all_top_idx = pd.concat([top_pc1_plot, top_pc2_plot]).index.unique()[:12]
plot_load = loadings.loc[all_top_idx, ['PC1', 'PC2']]
xp = np.arange(len(plot_load))
w = 0.35
axes[1].bar(xp - w/2, plot_load['PC1'].abs(), w, label='PC1', color=COLORS['purple'], alpha=0.75)
axes[1].bar(xp + w/2, plot_load['PC2'].abs(), w, label='PC2', color=COLORS['teal'], alpha=0.75)
axes[1].set_xticks(xp)
axes[1].set_xticklabels([c[:14] for c in plot_load.index], rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('|Loading|')
axes[1].set_title('Top Feature Contributions (PC1 & PC2)')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/figures/fig03_pca_variance.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig03_pca_variance.png")

# Hình 4: Pairplot 4 PC đầu
income_labels = pd.qcut(y, q=3, labels=['Low\n(<4K)', 'Mid\n(4-7K)', 'High\n(>7K)'])
point_colors = [COLORS['income'][l] for l in income_labels]
pc_labels = ['PC1', 'PC2', 'PC3', 'PC4']

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle('Hình 4: Pairplot – 4 Principal Components (màu = MonthlyIncome)', fontsize=13, fontweight='bold')

for i in range(4):
    for j in range(4):
        ax = axes[i][j]
        if i == j:
            for label, color in COLORS['income'].items():
                mask = income_labels == label
                ax.hist(X_pca[mask, i], bins=20, color=color, alpha=0.5)
            ax.set_xlabel(pc_labels[i], fontsize=9)
        elif i < j:
            ax.scatter(X_pca[:, j], X_pca[:, i], c=point_colors, s=8, alpha=0.4)
            ax.set_xlabel(pc_labels[j], fontsize=9)
            ax.set_ylabel(pc_labels[i], fontsize=9)
        else:
            ax.set_visible(False)

legend_elements = [plt.Rectangle((0,0),1,1, facecolor=c, label=l) for l, c in COLORS['income'].items()]
fig.legend(handles=legend_elements, loc='lower right', fontsize=10, title='MonthlyIncome')
plt.tight_layout()
plt.savefig('outputs/figures/fig04_pca_pairplot.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig04_pca_pairplot.png")

# Lưu PCA data
df_pca = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(n_vis)])
df_pca['MonthlyIncome'] = y
df_pca['JobLevel'] = job_level
df_pca.to_csv('data/processed/df_pca.csv', index=False)
print(f"✓ Saved: data/processed/df_pca.csv (shape: {df_pca.shape})")

# ═══════════════════════════════════════════════════════════
# BƯỚC 4 – t-SNE (Tuning Perplexity)
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 4 – t-SNE (Tuning Perplexity = 20, 30, 50)")
print("=" * 60)
print("⚠ Lưu ý: t-SNE chạy hơi lâu với 1470 samples, vui lòng chờ...")

point_colors_jl = [COLORS['jl'][l] for l in job_level]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Hình 5: t-SNE 2D – So sánh Perplexity (màu = JobLevel)', fontsize=13, fontweight='bold')

best_tsne = None
for idx, perp in enumerate([20, 30, 50]):
    print(f"  Đang chạy perplexity = {perp}...", flush=True)
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, max_iter=1000)
    X_t = tsne.fit_transform(X)
    axes[idx].scatter(X_t[:, 0], X_t[:, 1], c=point_colors_jl, s=12, alpha=0.5)
    axes[idx].set_title(f'Perplexity = {perp}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('t-SNE 1')
    axes[idx].set_ylabel('t-SNE 2')
    if perp == 30:
        best_tsne = X_t

legend_elements = [plt.Rectangle((0,0),1,1, facecolor=c, label=f'JobLevel {l}') for l, c in COLORS['jl'].items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=5, fontsize=10, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig('outputs/figures/fig05_tsne_perplexity.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig05_tsne_perplexity.png")
print("  → Chọn perplexity = 30 (cân bằng giữa cấu trúc cục bộ và toàn cục)")

# Lưu t-SNE data
df_tsne = pd.DataFrame(best_tsne, columns=['tSNE_1', 'tSNE_2'])
df_tsne['MonthlyIncome'] = y
df_tsne['JobLevel'] = job_level
df_tsne.to_csv('data/processed/df_tsne.csv', index=False)
print(f"✓ Saved: data/processed/df_tsne.csv (shape: {df_tsne.shape})")

# ═══════════════════════════════════════════════════════════
# BƯỚC 5 – CORRELATION HEATMAP & SCATTER PLOTS
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 5 – CORRELATION ANALYSIS")
print("=" * 60)

numeric_orig = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
corr_with_income = (df_cleaned[numeric_orig].corr()[TARGET]
                    .drop(TARGET)
                    .sort_values(key=abs, ascending=False))

print("\nTop 10 features tương quan với MonthlyIncome:")
for feat, val in corr_with_income.head(10).items():
    direction = "↑ thuận" if val > 0 else "↓ nghịch"
    print(f"  {feat:<35}: r = {val:+.4f}  ({direction})")

# Hình 6: Correlation bar chart + heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Hình 6: Tương quan Features với MonthlyIncome', fontsize=13, fontweight='bold')

# Bar chart
top15 = corr_with_income.head(15)
col_bars = [COLORS['teal'] if v > 0 else COLORS['coral'] for v in top15.values]
axes[0].barh(range(len(top15)), top15.values, color=col_bars, alpha=0.8)
axes[0].set_yticks(range(len(top15)))
axes[0].set_yticklabels([c[:22] for c in top15.index], fontsize=9)
axes[0].set_xlabel('Pearson Correlation')
axes[0].set_title('Top 15 Features vs MonthlyIncome')
axes[0].axvline(0, color='black', linewidth=0.5)

# Heatmap
top10_cols = list(corr_with_income.head(9).index) + [TARGET]
corr_matrix = df_cleaned[top10_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, ax=axes[1], mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, square=True, linewidths=0.5,
            annot_kws={'size': 8},
            xticklabels=[c[:12] for c in corr_matrix.columns],
            yticklabels=[c[:12] for c in corr_matrix.index])
axes[1].set_title('Correlation Matrix (Top 9 Features + Target)')
axes[1].tick_params(labelsize=8, rotation=45)

plt.tight_layout()
plt.savefig('outputs/figures/fig06_correlation.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig06_correlation.png")

# Hình 7: Scatter plots của top 4 features
top4 = list(corr_with_income.head(4).index)
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Hình 7: Scatter – Top 4 Features vs MonthlyIncome', fontsize=13, fontweight='bold')

for i, col in enumerate(top4):
    r_val = corr_with_income[col]
    axes[i].scatter(df_cleaned[col], y, s=15, alpha=0.3, color=COLORS['purple'])
    # Vẽ đường hồi quy
    z = np.polyfit(df_cleaned[col].values, y, 1)
    p = np.poly1d(z)
    x_line = np.linspace(df_cleaned[col].min(), df_cleaned[col].max(), 100)
    axes[i].plot(x_line, p(x_line), 'r--', lw=1.5, label='Trend')
    axes[i].set_xlabel(col, fontsize=10)
    axes[i].set_ylabel('MonthlyIncome' if i == 0 else '')
    axes[i].set_title(f'{col}\nr = {r_val:.3f}', fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.savefig('outputs/figures/fig07_scatter_top4.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig07_scatter_top4.png")

# ═══════════════════════════════════════════════════════════
# BƯỚC 6 – SO SÁNH PCA vs t-SNE
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 6 – SO SÁNH PCA vs t-SNE (cho báo cáo)")
print("=" * 60)

comparison = {
    'Phương pháp': ['PCA', 't-SNE'],
    'Loại': ['Tuyến tính', 'Phi tuyến'],
    'Bảo tồn cấu trúc': ['Toàn cục (global)', 'Cục bộ (local)'],
    'Tốc độ': ['Nhanh', 'Chậm (O(n²))'],
    'Reproducibility': ['Deterministic', 'Stochastic'],
    'Variance explained': ['Có thể tính', 'Không có ý nghĩa'],
    'Phù hợp với': ['Giảm chiều cho model', 'Visualization 2D'],
    'Kết quả với IBM HR': ['PC1 ≈ JobLevel', 'Cụm theo JobLevel rõ']
}
df_compare = pd.DataFrame(comparison)
print(df_compare.to_string(index=False))
all
print("\n📌 KẾT LUẬN:")
print("  • PCA phù hợp để giảm chiều trước khi huấn luyện model regression")
print("  • t-SNE phù hợp để visualization và kiểm tra cấu trúc cụm")
print("  • Cả hai đều xác nhận JobLevel là feature quan trọng nhất")

# ═══════════════════════════════════════════════════════════
# TÓM TẮT CUỐI
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📊 TÓM TẮT TUẦN 1 – EDA")
print("=" * 60)
print(f"""
  PCA       : {n_vis} components giải thích {pca_vis.explained_variance_ratio_.sum()*100:.1f}% variance
  t-SNE     : perplexity = 30 được chọn
  Top feature: JobLevel (r = {corr_with_income['JobLevel']:.3f})
  
  Output files:
    📁 data/processed/df_pca.csv
    📁 data/processed/df_tsne.csv
    📁 outputs/figures/ (fig01 → fig07)
""")
print("=" * 60)
print("✅ notebook_02_eda - HOÀN THÀNH!")

NOTEBOOK 02 – EDA & GIẢM CHIỀU DỮ LIỆU

📂 Đang đọc dữ liệu...
✓ Loaded: X=(1470, 46), y=(1470,)
✓ Target range: [1009, 19999], mean=6503

BƯỚC 1 – MISSING VALUES & TARGET DISTRIBUTION
✓ Completeness min: 100.0% → dataset rất sạch
✓ MonthlyIncome skewness: 1.368 (lệch phải – phân phối lương điển hình)
✓ Saved: outputs/figures/fig01_missing_target.png

BƯỚC 2 – OUTLIER BOXPLOTS (Phát hiện outliers bằng IQR)
✓ Saved: outputs/figures/fig02_outliers_boxplot.png

BƯỚC 3 – PCA (Principal Component Analysis)

Số components cần để đạt ngưỡng variance:
  80% → 24 components (52% số features)
  90% → 29 components (63% số features)
  95% → 33 components (72% số features)

Top 5 features đóng góp nhiều nhất vào PC1:
  TotalWorkingYears             : |loading| = 0.3907
  YearsAtCompany                : |loading| = 0.3852
  JobLevel                      : |loading| = 0.3820
  YearsInCurrentRole            : |loading| = 0.3344
  YearsWithCurrManager          : |loading| = 0.3277
✓ Saved: outputs/figu